# 04 — Baseline Models

## Objective

This notebook establishes baseline performance for the readmission prediction task using the leakage-safe, patient-level data split and preprocessing pipeline from Notebook 03.

Six standard classifiers are trained and compared on the validation set:
- Logistic Regression (linear baseline)
- Decision Tree (non-linear, interpretable baseline)
- Random Forest (bagging ensemble)
- XGBoost, LightGBM, CatBoost (gradient boosting ensembles)

### Evaluation Protocol

- The **validation set** is used exclusively for model comparison and selection.
- The **test set** is held out and will only be used once, at the very end of the project, for final unbiased evaluation.
- ROC-AUC is the primary selection metric, chosen for its robustness to the moderate class imbalance in this dataset (~53% / ~47%).
- Accuracy, Precision, Recall, and F1 are reported as supporting metrics.

### Reproducibility

All models are trained with `random_state = 42` for reproducible results.

In [30]:
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

baseline_data = joblib.load(MODELS_DIR / "baseline_data.pkl")

X_train_processed = baseline_data["X_train"]
X_val_processed = baseline_data["X_val"]
X_test_processed = baseline_data["X_test"]

y_train = baseline_data["y_train"]
y_val = baseline_data["y_val"]
y_test = baseline_data["y_test"]

print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

Train: (63444, 2304)
Validation: (15775, 2304)
Test: (20124, 2304)


## Data Validation

Before training, the loaded data is validated against the expected outputs of Notebook 03 — this guards against accidentally training on a stale or mismatched artifact.

In [31]:
assert X_train_processed.shape[0] == y_train.shape[0]
assert X_val_processed.shape[0] == y_val.shape[0]
assert X_test_processed.shape[0] == y_test.shape[0]
assert X_train_processed.shape[1] == X_val_processed.shape[1] == X_test_processed.shape[1]

print("Shape consistency: OK")

print("\nTarget distribution:")
for name, y in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    dist = y.value_counts(normalize=True).sort_index().round(4)
    print(f"\n{name}:")
    print(dist)

Shape consistency: OK

Target distribution:

Train:
readmitted
0    0.5301
1    0.4699
Name: proportion, dtype: float64

Validation:
readmitted
0    0.5324
1    0.4676
Name: proportion, dtype: float64

Test:
readmitted
0    0.5218
1    0.4782
Name: proportion, dtype: float64


## Baseline Model Training

Six standard classifiers are instantiated and trained on the preprocessed training data, then evaluated on the validation set using five metrics.

ROC-AUC is the primary selection criterion. All models use `random_state = 42` for reproducibility; where applicable, `n_jobs = -1` is used to parallelize training.

In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
import time

models = {
    "Logistic Regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostClassifier(random_state=RANDOM_STATE, verbose=0),
}

def evaluate_model(name, model):
    start = time.time()
    model.fit(X_train_processed, y_train)
    train_time = time.time() - start

    y_pred = model.predict(X_val_processed)
    y_prob = model.predict_proba(X_val_processed)[:, 1]

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred),
        "Recall": recall_score(y_val, y_pred),
        "F1": f1_score(y_val, y_pred),
        "ROC-AUC": roc_auc_score(y_val, y_prob),
        "Train Time (s)": round(train_time, 2),
    }

results_list = []
for name, model in models.items():
    print(f"Training {name}...")
    results_list.append(evaluate_model(name, model))

results_df = pd.DataFrame(results_list).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print("BASELINE PERFORMANCE ON VALIDATION SET")
print("="*70)
print(results_df.round(4).to_string(index=False))

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training XGBoost...
Training LightGBM...
Training CatBoost...

BASELINE PERFORMANCE ON VALIDATION SET
              Model  Accuracy  Precision  Recall     F1  ROC-AUC  Train Time (s)
           CatBoost    0.6378     0.6312  0.5422 0.5833   0.6918           28.01
           LightGBM    0.6359     0.6257  0.5510 0.5860   0.6889            0.82
            XGBoost    0.6375     0.6296  0.5463 0.5850   0.6845            1.05
      Random Forest    0.6295     0.6258  0.5170 0.5662   0.6780           71.34
Logistic Regression    0.6279     0.6244  0.5125 0.5630   0.6678            4.95
      Decision Tree    0.5524     0.5215  0.5199 0.5207   0.5504           31.76


## Save Baseline Results & Register Candidate Models

CatBoost (ROC-AUC = 0.6918) and LightGBM (ROC-AUC = 0.6889) are registered as **champion candidates** — they will carry forward through feature engineering, feature selection, and hyperparameter optimization.

XGBoost (ROC-AUC = 0.6845) is retained as a **reference model** in all future comparison tables, but is not itself subjected to feature selection or optimization. This keeps the experimental scope focused while still providing a sanity check on whether the champions' improvements are meaningful.

Note on this baseline vs. the pre-cleaning baseline: removing expired/hospice encounters (Notebook 03) lowered CatBoost's validation ROC-AUC from 0.7081 to 0.6918. This is expected and desired — the earlier score partly reflected the model exploiting a mortality-related label artifact rather than genuine readmission risk.

All results are saved to `reports/` as the reference point for every future comparison.

In [33]:
# Save the full baseline results table
baseline_results_path = REPORTS_DIR / "baseline_results.csv"
results_df.to_csv(baseline_results_path, index=False)

# Start the running comparison log used across all future notebooks
comparison_log_path = REPORTS_DIR / "model_comparison_log.csv"

log_df = results_df.copy()
log_df.insert(0, "stage", "01_baseline")
log_df.insert(1, "notebook", "04_baseline_models")

log_df.to_csv(comparison_log_path, index=False)

print(f"Baseline results saved to: {baseline_results_path}")
print(f"Comparison log saved to:  {comparison_log_path}")

# Register candidate models explicitly
CHAMPION_MODELS = ["CatBoost", "LightGBM"]
REFERENCE_MODEL = "XGBoost"

print("\n" + "="*70)
print("CHAMPION CANDIDATES (will be optimized in future notebooks)")
print("="*70)
print(results_df[results_df["Model"].isin(CHAMPION_MODELS)].round(4).to_string(index=False))

print("\n" + "="*70)
print(f"REFERENCE MODEL (tracked but not optimized): {REFERENCE_MODEL}")
print("="*70)
print(results_df[results_df["Model"] == REFERENCE_MODEL].round(4).to_string(index=False))

# Save the fitted champion + reference models for reuse
joblib.dump(models["CatBoost"], MODELS_DIR / "baseline_catboost.pkl")
joblib.dump(models["LightGBM"], MODELS_DIR / "baseline_lightgbm.pkl")
joblib.dump(models["XGBoost"], MODELS_DIR / "baseline_xgboost.pkl")

print("\nModels saved to:", MODELS_DIR)

Baseline results saved to: e:\all projects\Machine Learning\Advanced-ML-Hospital-Readmission\reports\baseline_results.csv
Comparison log saved to:  e:\all projects\Machine Learning\Advanced-ML-Hospital-Readmission\reports\model_comparison_log.csv

CHAMPION CANDIDATES (will be optimized in future notebooks)
   Model  Accuracy  Precision  Recall     F1  ROC-AUC  Train Time (s)
CatBoost    0.6378     0.6312  0.5422 0.5833   0.6918           28.01
LightGBM    0.6359     0.6257  0.5510 0.5860   0.6889            0.82

REFERENCE MODEL (tracked but not optimized): XGBoost
  Model  Accuracy  Precision  Recall    F1  ROC-AUC  Train Time (s)
XGBoost    0.6375     0.6296  0.5463 0.585   0.6845            1.05

Models saved to: e:\all projects\Machine Learning\Advanced-ML-Hospital-Readmission\models


## Summary

Six baseline classifiers were trained on the corrected patient-level split (63,444 train / 15,775 validation / 20,124 test), using the preprocessing pipeline from Notebook 03 (44 raw features → 2,304 features after one-hot encoding).

CatBoost achieved the best validation ROC-AUC (0.6918), followed closely by LightGBM (0.6889) and XGBoost (0.6845). Random Forest, Logistic Regression, and Decision Tree trailed further behind.

**CatBoost and LightGBM are selected as champion candidates** and will be carried through feature engineering, feature selection, and hyperparameter optimization. XGBoost is retained as a reference baseline for comparison but will not itself be optimized, keeping the experimental scope focused.

Results are logged in `reports/baseline_results.csv` and `reports/model_comparison_log.csv`.

**Next notebook (05):** Feature engineering — including clinical grouping of the high-cardinality ICD-9 diagnosis codes (`diag_1/2/3`), which are the main contributor to the current 2,304-dimensional sparse feature space.